<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Ask the Hexagons: Questions in Words, Answers from the Table

> **Optional notebook.** It adds a language model on top of the database from
> `05-INT-Hex_Integration.ipynb`. Nothing else in the workshop depends on it. You choose how the
> language model is reached: a free Gemini key, an open model on the Colab GPU, or a chat window
> that you copy and paste into.

**The question:** can you ask the ward a question in plain words, and get an answer that you can
check? "Which hexagons are hot and have no public green?" is easy to say and takes a few lines of
SQL to answer. A language model can write those lines for you. The objective of this notebook is to
try this, and to see where it goes wrong.

**What the model does, and what it does not.** The model never calculates. It reads your question,
the names of the columns and the legend, and it **writes one SQL query**. DuckDB runs the query on
the table and returns the numbers. The model then explains the first rows of the result in a few
sentences, and first checks that the rows really match your question. You always see the SQL, so you can check what was asked.

```
your question  ->  model writes SQL  ->  DuckDB runs it  ->  table of hexagons  ->  model explains
                   (you see it)          (read-only)          (the numbers)
```

**What leaves your notebook.** It depends on `PROVIDER` (Section 1):

- `"gemini"`: the text of the prompt (the column names and the legend), your question, and the first
  20 rows of each result are sent to Google. These are numbers per hexagon from open data. On the
  free tier, Google states that it may use the content to improve its products. The database itself
  is not sent.
- `"local"`: nothing is sent to a model provider. The model runs on the Colab GPU.
- `"paste"`: only what you copy into the chat window yourself.

Check the terms of the service and your ethics rules before you use it with students.

**Library:** [`sitex`](https://github.com/ArchiColab/sitex). This notebook installs its own extra
packages in Section 1 and does not need the `sitex` library.

**Requires (run first):** `05-INT-Hex_Integration.ipynb`, which builds `{slug}_hex.duckdb`.

**Optional:** `{slug}_provenance.csv` (written by Section 13 of `00-Data-Acquisition.ipynb`). If it is there, you can also
ask where the data comes from: its date, owner, resolution, licence and coordinate system.

**Output (`data/gdf/`):** `{slug}_ask_log.csv`, one row per question: the question, how the model read it, the
SQL, whether it ran, the number of rows and the tokens used.

**What you edit:** `PROVIDER` and `SLUG` in Section 1, and your own question in Section 5.

---
## 1. Install & choose the model

Choose with `PROVIDER`:

| `PROVIDER` | What happens | What it costs |
|---|---|---|
| `"gemini"` | The notebook calls a Google Gemini model. You need a **free API key** from Google AI Studio. | Free, within the limits of the free tier (shown in AI Studio). The content you send may be used by Google to improve its products. |
| `"local"` | An open model from Hugging Face (`Qwen/Qwen2.5-Coder-7B-Instruct`, Apache 2.0 licence) is downloaded and run on the Colab GPU. No key. | Needs a GPU runtime (Runtime > Change runtime type). The free GPU is not always available, the first download takes minutes in every new session, and a small model makes more SQL mistakes. |
| `"paste"` | No key, no GPU. The notebook prints the prompt, you copy it into any chat window (for example ChatGPT), and paste the SQL it gives back in Section 6. | Free. A little manual work. |

The Gemini call uses the plain web interface (the `requests` package that Colab already has); nothing else is installed for it.

`MODEL` can stay empty: each provider then uses a default. If Gemini rejects the default name, take
a current one from [the Gemini pricing page](https://ai.google.dev/gemini-api/docs/pricing).

**Keep the Gemini key secret.** Never type a key into a cell: a notebook that you share would contain it.
In Colab, click the key icon (Secrets) in the left bar, add a secret named `GEMINI_API_KEY`, paste your
key as the value, and switch on *Notebook access*. The notebook reads it from there and never prints it.

In [ ]:
#@title <font color=#1B7192> Configuration </font>  { display-mode: "form" }

import os
import sys
import time
from pathlib import Path

PROVIDER = "gemini" #@param ["gemini", "local", "paste"]
#@markdown <font color=#1B7192> "gemini": free Google key. "local": open model on the Colab GPU. "paste": copy the prompt into a chat window yourself. </font>
MODEL = "" #@param {type:"string"}
#@markdown <font color=#1B7192> Leave empty for the default of the provider. Not used for "paste". </font>
SLUG = "pleiku" #@param {type:"string"}

DEFAULT_MODEL = {"gemini": "gemini-3.5-flash-lite", "local": "Qwen/Qwen2.5-Coder-7B-Instruct"}
MODEL = MODEL.strip() or DEFAULT_MODEL.get(PROVIDER, "")

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install -q duckdb
    if PROVIDER == "local":
        %pip install -q transformers accelerate bitsandbytes

    from google.colab import drive
    drive.mount("/content/gdrive")
    GDF_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs/gdf")
else:
    GDF_DIR = Path("..") / "data" / "gdf"

import duckdb
import pandas as pd
from IPython.display import display

print("DuckDB", duckdb.__version__)

---
## 2. Open the database

The database from `05-INT-Hex_Integration.ipynb` is opened **read-only**, and reading other files
is switched off, so a query can only read the tables in it. In Colab the file is copied to the
local disk first, because the Google Drive folder is slow.

In [ ]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 2. Open the database ###", flush=True)

DB_PATH = GDF_DIR / f"{SLUG}_hex.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"{DB_PATH} not found -- run 05-INT-Hex_Integration.ipynb first.")
if IN_COLAB:
    import shutil
    shutil.copy(DB_PATH, "/content/hex.duckdb")
    DB_PATH = Path("/content/hex.duckdb")

con = duckdb.connect(str(DB_PATH), read_only=True, config={"enable_external_access": False})

# the data sources record (optional): one row per data layer, so that questions about where the data comes from can be answered
PROV_PATH = GDF_DIR / f"{SLUG}_provenance.csv"
has_prov = PROV_PATH.exists()
if has_prov:
    provenance = pd.read_csv(PROV_PATH, dtype=str, encoding="utf-8-sig").fillna("")
    con.register("provenance", provenance)          # in memory only: the database file stays read-only
else:
    print(f"No {PROV_PATH.name}: questions about the data sources are not possible (run Section 13 of 00-Data-Acquisition.ipynb).")

tables = sorted(r[0] for r in con.sql("SHOW TABLES").fetchall())   # the view hex_all is listed too
n_hex = con.sql("SELECT count(*) FROM hex_all").fetchone()[0]
print("Tables:", ", ".join(tables))
print(f"hex_all: {n_hex} hexagons" + (f"; provenance: {len(provenance)} rows" if has_prov else ""))

print(f"--- 2. done in {time.time() - _t0:.2f}s ---\n")

---
## 3. The prompt: columns, legend and rules

The model knows nothing about your ward. Everything it knows comes from this text:

1. **The columns** of the view `hex_all`, with the unit and the definition from the legend. (A column that exists in two tables, such as `snap_dist_m`, has the table name in front.) If the data sources record is there, also the columns of the table `provenance`.
2. **The rules.** An empty value is not zero. The numbers are per hexagon, not per resident. If the columns cannot answer the question, say so instead of guessing.
3. **How to read vague words.** The model must start its query with one comment line, `-- Reading: ...`, that says how it understood words such as *hot*, *dense* or *worst* and which thresholds it chose. When your question joins two conditions with "and", the query must require **both**, each as a threshold (the top quarter for *high*, the bottom quarter for *low*). The quarter is a default of this notebook, not a standard: change it in the rules below if you want another.

The `limits` column of the legend is not in the first prompt, to keep it short. It is added to the
second prompt, where the model explains the result, for the columns that the query used.

Read the prompt that is printed below. It is the whole "knowledge" of the model.

In [ ]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 3. The prompt ###", flush=True)

legend = con.sql("""
    SELECT table_name, column_name, unit, definition, limits FROM legend
    WHERE table_name LIKE 'hex_%' AND column_name <> 'geometry'
""").df()

names = legend["column_name"].value_counts()
lines = []
for table, column, unit, definition in legend[["table_name", "column_name", "unit", "definition"]].itertuples(index=False):
    if column == "h3_id" and table != "hex_grid":
        continue                                           # the key is listed once
    if names[column] > 1 and column != "h3_id":           # same name in two tables: the view puts the table in front
        column = f"{table.removeprefix('hex_')}_{column}"
    lines.append(f"- {column} ({unit}): {definition}")
COLUMN_LIST = "\n".join(lines)

PROV_BLOCK = """

Columns of provenance (all text):
- layer: name of the data layer; kind: raster, vector, derived or crs
- dataset: the dataset and its version; provider: owner or provider
- data_date: date or year of the data; native_resolution: resolution of the dataset; file_resolution: resolution of the saved file
- file_crs: CRS of the saved file (EPSG:32649 is the UTM zone in metres, EPSG:4326 is longitude and latitude in degrees)
- licence and attribution: terms of use; used_for: the hexagon columns it feeds; retrieved: date the file was saved; notes""" if has_prov else ""


def rules_text():
    if has_prov:
        head = ("You write ONE DuckDB SELECT query. There are two sources: the view hex_all (one row per hexagon of a city ward) and "
                "the table provenance (one row per data layer or coordinate system: where the data comes from).")
        prov_rule = ("\n- Questions about where data comes from, its date, resolution, owner, licence or CRS use the table provenance; "
                     "questions about places, values or hexagons use hex_all. Never join them. In provenance, kind is raster, vector, derived "
                     "or crs, and a value that contains [TO FILL] is unknown. For text searches use ILIKE with a whole word or phrase, "
                     "for example ILIKE '%Landsat%'.")
    else:
        head = "You write ONE DuckDB SELECT query on the view hex_all. It has one row per hexagon of a city ward."
        prov_rule = ""
    return f"""{head}
Return only the SQL, with no markdown. The first line must be an SQL comment that starts with '-- Reading:' and says in one sentence how you understood the vague words of the question (hot, dense, worst, good ...) and which thresholds you chose.
Rules:
- An empty value (NULL) means 'not reachable' or 'no data', never zero. Test it with IS NULL.
- All numbers in hex_all are per hexagon, not per resident. There is no population column.
- If the columns cannot answer the question, return: SELECT 'cannot be answered with these columns' AS note
- If the question joins conditions with 'and' (for example hottest and most densely built), the WHERE clause must require ALL of them. Give each vague word a threshold: for high, hottest or most use the top quarter, quantile_cont(column, 0.75); for low or least use the bottom quarter, quantile_cont(column, 0.25). Do not rank by one column and use the other only to break ties.{prov_rule}
- Use only the columns listed below."""


def build_prompt(question):
    return f"{rules_text()}\n\nColumns of hex_all:\n{COLUMN_LIST}{PROV_BLOCK}\n\nQuestion: {question}"

example = build_prompt("Which 5 hexagons have the highest land surface temperature?")
print(example)
print(f"\n({len(example):,} characters, roughly {len(example) // 4:,} tokens)")

print(f"--- 3. done in {time.time() - _t0:.2f}s ---\n")

---
## 4. The model, and the function `ask`

This cell connects to the model (nothing is sent yet; with `"local"` it downloads and loads the model,
which takes a few minutes) and defines `ask(question)`, which does the steps and keeps the
safeguards:

- only **one SELECT query** is run (nothing else is accepted);
- the database is read-only, so nothing can be changed;
- the model's **reading** of your question and the SQL are printed **before** the result, so you can check them;
- the model sees at most 20 rows of the result; before explaining, it checks that the rows meet every
  condition of your question, and the explanation uses the `limits` of the columns that the query used;
- Gemini and the local model are asked for the same answer each time as far as possible (temperature 0), but the
  same question can still give a different query.

Every question is added to the log file, with the tokens used. The key is never printed or written to the log.

In [ ]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 4. The model and the function ask ###", flush=True)

import re
from datetime import datetime

import requests

LAST_USAGE = {"in": None, "out": None}      # tokens of the last model call (filled by call_llm)

call_llm = None
if PROVIDER == "gemini":
    API_KEY = None
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = os.environ.get("GEMINI_API_KEY")
    if not API_KEY:
        raise ValueError("No key found. In Colab: click the key icon (Secrets), add GEMINI_API_KEY, paste your key "
                         "and switch on Notebook access. Or set PROVIDER to 'local' or 'paste'.")
    GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

    def call_llm(prompt):
        try:
            reply = requests.post(GEMINI_URL, params={"key": API_KEY}, timeout=120,
                                  json={"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"temperature": 0}})
            data = reply.json()
        except requests.RequestException as exc:      # the message of the exception would contain the key
            raise RuntimeError(f"Network error ({type(exc).__name__}).") from None
        if not reply.ok:
            raise RuntimeError(data.get("error", {}).get("message", f"Gemini error {reply.status_code}"))
        usage = data.get("usageMetadata")
        LAST_USAGE.update({"in": usage.get("promptTokenCount") if usage else None,
                           "out": ((usage.get("candidatesTokenCount") or 0) + (usage.get("thoughtsTokenCount") or 0)) if usage else None})
        parts = (data.get("candidates") or [{}])[0].get("content", {}).get("parts", [])
        if not parts:
            raise RuntimeError("The model returned no answer (blocked or empty).")
        return "".join(part.get("text", "") for part in parts)

    print(f"Connected: Gemini, model {MODEL}")

elif PROVIDER == "local":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not torch.cuda.is_available():
        print("No GPU found. In Colab: Runtime > Change runtime type > GPU, then run the notebook again. "
              "Or set PROVIDER to 'gemini' or 'paste'.")
    else:
        tokenizer = AutoTokenizer.from_pretrained(MODEL)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
            device_map="auto",
        )

        def call_llm(prompt, max_new_tokens=400):
            text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                                 add_generation_prompt=True)
            inputs = tokenizer([text], return_tensors="pt").to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            n_in = inputs["input_ids"].shape[1]
            LAST_USAGE.update({"in": n_in, "out": out.shape[1] - n_in})
            return tokenizer.decode(out[0][n_in:], skip_special_tokens=True)

        print(f"Model loaded: {MODEL}")

else:
    print("PROVIDER is 'paste': no model is called. ask() prints the prompt for you to copy.")

LOG_PATH = GDF_DIR / f"{SLUG}_ask_log.csv"
LOG_COLUMNS = ["time", "model", "question", "reading", "sql", "ran", "n_rows",
               "tokens_in_1", "tokens_out_1", "tokens_in_2", "tokens_out_2", "error"]
if LOG_PATH.exists() and LOG_PATH.read_text(encoding="utf-8").splitlines()[0] != ",".join(LOG_COLUMNS):
    LOG_PATH.replace(LOG_PATH.with_name(LOG_PATH.stem + "_old.csv"))       # an older log with other columns is kept apart
MODEL_NAME = f"{PROVIDER}:{MODEL}" if call_llm else "pasted"


def split_reading(text):
    """The leading '-- Reading: ...' comment lines of a model answer, and the SQL after them."""
    rows = re.sub(r"```(?:sql)?", "", text).strip().split("\n")
    reading = []
    while rows and re.match(r"\s*--", rows[0]):
        reading.append(re.sub(r"^\s*--\s*(Reading:)?\s*", "", rows.pop(0), flags=re.I).strip())
    return " ".join(reading), "\n".join(rows)


def clean_sql(text):
    """The SQL from a model answer: no code fences, one statement, SELECT (or WITH) only."""
    text = re.sub(r"```(?:sql)?", "", text).strip().rstrip(";").strip()
    if ";" in text or not re.match(r"(?is)^(select|with)\b", text):
        raise ValueError("Only one SELECT query is accepted.")
    return text


def log_entry(entry):
    row = pd.DataFrame([{**{c: "" for c in LOG_COLUMNS}, **entry}])[LOG_COLUMNS]
    row.to_csv(LOG_PATH, mode="a", header=not LOG_PATH.exists(), index=False, encoding="utf-8")


def run_sql(question, sql_text):
    """Run the SQL, show its reading, the SQL and the result. Returns (table or None, sql, log entry)."""
    reading, body = split_reading(sql_text)
    sql, table, error = None, None, ""
    try:
        sql = clean_sql(body)
        table = con.sql(sql).df()
    except Exception as exc:
        error = str(exc).splitlines()[0][:300]
    if reading:
        print("How the model read your question: " + reading + "\n")
    print("SQL:\n" + (sql or body.strip()))
    if error:
        print(f"\nThe query did not run: {error}")
    else:
        print(f"\n{len(table)} row(s):")
        display(table.head(20))
    entry = {"time": datetime.now().isoformat(timespec="seconds"), "model": MODEL_NAME, "question": question, "reading": reading,
             "sql": sql or body.strip(), "ran": error == "", "n_rows": "" if error else len(table), "error": error}
    return table, sql, entry


def explain(question, sql, table):
    only_prov = bool(re.search(r"\bprovenance\b", sql, re.I)) and not re.search(r"\bhex_all\b", sql, re.I)
    used = legend[legend["column_name"].apply(lambda c: c in sql)]
    limits = "\n".join(f"- {c}: {l}" for c, l in used[["column_name", "limits"]].drop_duplicates().itertuples(index=False))
    unit_note = ("State dates, owners, resolutions, coordinate systems and licence terms exactly as they appear in the result; [TO FILL] means unknown. "
                 if only_prov else "Say that the numbers are per hexagon, not per resident. ")
    return call_llm(
        "Explain this result to a student in at most 4 sentences. Use only the result below. "
        "First check whether the rows meet every condition of the question; if some do not (for example a hexagon with 0 buildings "
        "in a question about dense building), say so in the first sentence. "
        + unit_note + "Mention a limit from the list if it matters. "
        "If the result is empty, say that no hexagon matches.\n\n"
        f"Question: {question}\nSQL: {sql}\nResult (first rows):\n{table.head(20).to_string()}\n\nLimits:\n{limits}"
    )


def ask(question):
    """With a model: question -> SQL -> result -> explanation. With 'paste': prints the prompt (see Section 6)."""
    if call_llm is None:
        print("Copy this prompt into a chat window, then paste the SQL it returns into Section 6:\n")
        print(build_prompt(question))
        return None
    try:
        sql_text = call_llm(build_prompt(question))
    except Exception as exc:
        print(f"The model call failed: {str(exc).splitlines()[0][:300]}")
        return None
    usage1 = dict(LAST_USAGE)
    table, sql, entry = run_sql(question, sql_text)
    entry.update({"tokens_in_1": usage1["in"] if usage1["in"] is not None else "", "tokens_out_1": usage1["out"] if usage1["out"] is not None else ""})
    if table is not None:
        try:
            print("\nExplanation:\n" + explain(question, sql, table))
            entry.update({"tokens_in_2": LAST_USAGE["in"] if LAST_USAGE["in"] is not None else "",
                          "tokens_out_2": LAST_USAGE["out"] if LAST_USAGE["out"] is not None else ""})
        except Exception as exc:
            print(f"The explanation failed: {str(exc).splitlines()[0][:300]}")
    if usage1["in"] is not None:
        print(f"\nTokens: step 1 in {usage1['in']} / out {usage1['out']}; step 2 in {entry.get('tokens_in_2') or '?'} / out {entry.get('tokens_out_2') or '?'}")
    log_entry(entry)
    return table

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")

---
## 5. Ask questions

Six example questions, then four about the street network and the data sources, then your own. After each
answer, check three things (Section 7). The sixth example asks for something the table does not have: a good
answer says so. Read the line "How the model read your question" every time: the same words can be read in
more than one way, and it is the first place where a result can go wrong.

In [ ]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

EXAMPLES = [
    "Which 5 hexagons have the highest land surface temperature?",
    "How many hexagons have no public green within the walking time?",
    "Which hexagons are among the hottest 25% and have no public green within reach? Show the temperature and the number of buildings.",
    "How many hexagons have no facility within 15 minutes of walking?",
    "Which hexagon has the most buildings, and what is its dominant function?",
    "How many residents live in the hottest hexagon?",
]
for i, q in enumerate(EXAMPLES, 1):
    print(f"\n===== Question {i}: {q}")
    ask(q)

### The street network and the data sources

Two questions read the space syntax columns, as in the reading table of `03-NA01-SpaceSyntax.ipynb`: a **local
centre** is busy within 800 m but not on the main routes of the whole city (high NACH within 800 m, low NACH over the
whole network); a **through route** is the other way round. Two more questions go to the data sources table, if
you have it: they are about the layers, not about the hexagons. A text search in SQL can match more than you mean
(`ILIKE '%NC%'` also finds the word "include"): read the SQL.

In [ ]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

MORE_EXAMPLES = [
    "Which hexagons look like a local centre: busy within 800 m but not on the main routes of the whole city?",
    "Which hexagons look like a through route: little movement within 800 m but a lot across the whole city?",
]
if has_prov:
    MORE_EXAMPLES += [
        "Where does the surface temperature data come from, what date is it, and how fine is it?",
        "Which data layers are saved in longitude and latitude (EPSG:4326), and which in the UTM zone?",
    ]
for i, q in enumerate(MORE_EXAMPLES, 7):
    print(f"\n===== Question {i}: {q}")
    ask(q)

In [ ]:
#@title <font color=#1B7192> Your own question </font>  { display-mode: "form" }

MY_QUESTION = "Which hexagons have high street connectivity but few facilities within 15 minutes?" #@param {type:"string"}

ask(MY_QUESTION)

---
## 6. Without a key: paste the SQL

Only for `PROVIDER = "paste"`. Run `ask(...)` in Section 5: it prints the prompt. Copy the prompt into
a chat window, copy the SQL it returns into the cell below, and run it. The notebook checks that it
is one SELECT query, runs it read-only, and writes the log.

In [ ]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

QUESTION = "Which 5 hexagons have the highest land surface temperature?" #@param {type:"string"}
PASTED_SQL = """
SELECT 1
"""
table, sql, entry = run_sql(QUESTION, PASTED_SQL)
log_entry(entry)

---
## 7. Check the answer

A fluent answer can be wrong. For every question, check four things:

| Check | How |
|---|---|
| **Did the model read your question the way you meant it?** | Read the line "How the model read your question". Words such as *hot*, *dense* or *worst* got a threshold: is it the one you would choose? |
| **Does the SQL ask what you asked?** | Read the `WHERE` part against your question: the right column, the right direction (`>` or `<`), the right number, and **all** the conditions of your question. |
| **Is an empty value treated correctly?** | "No public green" is `access_public_min IS NULL`, not `= 0`. A zero means the junction lies inside a green space. |
| **Does the explanation stay within the table?** | It should say per hexagon, not per resident, and should not claim more than the columns can tell. Compare with the `limits` in the legend. |

Then compare with a query you write yourself in `05-INT-Hex_Integration.ipynb`. The log file
`{slug}_ask_log.csv` keeps every question, the model's reading, its SQL and the tokens used, so you can count how often the SQL ran, and
how often it was right.

---
## Notes

- **The model writes the query; DuckDB does the calculation.** The numbers in the table are exact.
  The words around them are the model's, and can be wrong.
- **A query that runs is not always the right query.** Smaller and cheaper models make more mistakes.
- **The same question can give a different query each time,** even at temperature 0. Run it three times, compare the readings and the SQL in the log, and count how often they agree.
- **The thresholds are a default.** The model is told to use the top or bottom quarter for vague words. That is a choice of this notebook, not a standard.
- **A Gemini key belongs to a person.** Keep it in Colab Secrets, never in a cell, and delete a key that
  has been shared by mistake. The free tier has limits per minute and per day; a class that asks at the
  same moment can reach them.
- **The local model is slow to start.** The Colab GPU is not always available, and the model has to be
  loaded again in every new session.
- **This is a trial, not a method.** Ten questions in one ward do not show how well it works
  elsewhere.

---
## When the model is wrong

Where the answer and the table disagree, write down why. Common reasons:

- **An empty value read as zero.** The query says `= 0` where it should say `IS NULL`.
- **A column that does not exist.** The model invents a name such as `walkability` or `population`.
  The query then fails, or it picks a similar column that means something else.
- **Only one of two conditions.** A question about the hottest *and* most densely built hexagons is answered by ranking on temperature alone, so a hexagon with 0 buildings comes first.
- **The right column, the wrong direction.** "Poor access" is a high walking time, or an empty one.
- **The explanation says more than the table.** For example it speaks about residents or about
  causes.

Write one sentence per mistake: *"I asked …, the model wrote …, it should have been …"*.

## Where to go next

| Next step | Open |
|---|---|
| Write the same queries yourself and read the legend | [`05-INT-Hex_Integration.ipynb`](05-INT-Hex_Integration.ipynb) |
| Go to the street behind a hexagon | [`02-ENV-UHI_Analysis.ipynb`](02-ENV-UHI_Analysis.ipynb), [`03-NA06-Green_Accessibility.ipynb`](03-NA06-Green_Accessibility.ipynb) |

**Into the design phase:** a question that you can ask in words and check in SQL is a question you can
put to a client or a colleague. Keep the log of the questions that gave a useful answer.